# The Prompt Generator Pattern - AI as a Prompt Engineer

Hello everyone. Today, we're exploring a fascinating \"meta\" technique where we use an AI to help us do our job as prompt engineers. So far, we have been carefully crafting prompts. But what if we could use the AI itself to build better prompts for us?

This is the **Prompt Generator Pattern**. The core idea is to use a powerful, state-of-the-art model to act as your expert prompt engineering partner. Its task is not to solve the final problem, but to **generate a high-quality, optimized prompt** that you can then use for your actual task.

This is especially useful when you want to use a cheaper, faster model for a repetitive production task but need the prompt to be perfectly optimized for reliability.

## Helper Functions

First, let's set up our standard helper functions.

In [ ]:
import litellm
from IPython.display import display, Markdown
from textwrap import dedent
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = "openai/gpt-4o-mini"
MAX_TOKENS_DEFAULT = 500

def get_completion(
    prompt,
    model=MODEL_NAME,
    max_tokens=MAX_TOKENS_DEFAULT,
    **kwargs
):
    if "gpt-5" in model:
        kwargs["max_completion_tokens"] = max_tokens
    else:
        kwargs["max_tokens"] = max_tokens
        
    parsed_messages = []

    if type(prompt) is str:
        parsed_messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]
    else:
        parsed_messages = prompt

    response = litellm.completion(
        model=model,
        messages=parsed_messages,
        **kwargs
    )

    return response.choices[0].message.content

print("Setup complete. Helper functions and code context are ready.")

## The Master Prompt (Generating a Prompt)

Our first step is to craft a \"master prompt.\" This prompt doesn't solve the final task; it *describes the prompt we want*. We will go ahead and implement a conversation flow to interactively craft our prompt. (You can also just do this exercise in ChatGPT, Claude Desktop, or any other AI application).

In [ ]:
master_prompt = [
    {
        "role": "system",
        "content": dedent("""
        You are an expert prompt engineer, and your task is to create a high-quailty,
        optimized system prompt based on a user's specification.

        ## Process:

        1. Start by asking me to provide the task I want you to generate a prompt for.
        2. Once I provide the task, ask me questions to clarify any doubts or missing information.
        3. Once you have the necessary information, creata a detailed and effective system prompt that
            I can use with an AI system to tackle the task at hankd.
        4. When I write "GENERATE", you will generate the final prompt based on the information we have discussed.

        ## Generated Prompt output rules:
        * Output only the text of the genereatead prompt.
        * Do not output any triple backticks nor code blocks, just the content of the generated prompt.
        * Leverage Markdown an dprompt engineering best practices to structure the prompt.
        """)
    }
]

